# ATML PA2 — Colab runner (thin wrapper, all code lives in the GitHub repo)

**State is shared across Colab accounts through one Google Drive folder.**

## One-time setup (do this once, from your main Google account)
1. In Google Drive create a folder named **`PA2_state`**.
2. Share it (Editor) with every other Google account you will run Colab from.
3. In each other account: open *Shared with me* → right-click `PA2_state` → **Organize → Add shortcut → My Drive**.

Every account then sees the same path `/content/drive/MyDrive/PA2_state`, which holds:
- `outputs/` — trained LoRA adapters (needed later by Task 4)
- `results/` — manifests, logs, metrics (returned to GitHub)

## Every session
1. Runtime → Change runtime type → **T4 GPU**.
2. Set `WORKER` below to a name for this account (e.g. `acct1`, `acct2`).
3. Run the **Setup** cells, then the experiment cell you want.
4. Look at the **status board** first — never start an experiment that another account shows as `running`.

Each experiment writes `status.json` (running/done/failed). Finished runs are skipped automatically,
a run with a fresh heartbeat on another account is refused, and a crashed run is archived (not overwritten) before retrying.

In [ ]:
# ---- Session settings ----
WORKER = "acct1"                      # <-- change per Google account
REPO_URL = "https://github.com/TalhaShoyo10/PA2_28100131.git"
REF = "main"                          # branch or exact commit SHA to run
STATE_DIR = "/content/drive/MyDrive/PA2_state"
REPO_DIR = "/content/PA2"

## Setup

In [ ]:
import os, subprocess
from google.colab import drive
drive.mount("/content/drive")
assert os.path.isdir(STATE_DIR), f"{STATE_DIR} not found - create it or add the shared shortcut to My Drive (see top)."
for sub in ("outputs", "results", "results/logs"):
    os.makedirs(f"{STATE_DIR}/{sub}", exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
if not os.path.isdir(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git fetch -q origin && git checkout -q {REF} && git pull -q --ff-only || true
!git log --oneline -1 && git status --short --untracked-files=no

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# Course assets (~1.07 GB, pinned revision) + validation + environment record + objective tests
!python -m scripts.download_assets
!python -m scripts.validate_assets
!python -m scripts.check_environment
!python -m pytest tests/ -q -p no:cacheprovider

In [ ]:
# Point outputs/ and results/ at the shared Drive folder
for sub in ("outputs", "results"):
    path = os.path.join(REPO_DIR, sub)
    if os.path.islink(path):
        continue
    if os.path.isdir(path):
        assert not os.listdir(path), f"{path} has local files; move them to Drive first"
        os.rmdir(path)
    os.symlink(f"{STATE_DIR}/{sub}", path)
os.environ["PA2_WORKER"] = WORKER
os.environ["PYTHONUNBUFFERED"] = "1"
!ls -la {REPO_DIR} | grep -E "outputs|results"

In [ ]:
import datetime, json, glob, time

def run(cmd, log_name):
    """Run a repo command, stream output, and keep a copy in results/logs on Drive."""
    ts = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
    log = f"results/logs/{log_name}__{WORKER}__{ts}.log"
    get_ipython().system(f"bash -c 'set -o pipefail; cd {REPO_DIR} && {cmd} 2>&1 | tee {log}'")
    code = get_ipython().user_ns.get("_exit_code", 0)
    if code:
        raise RuntimeError(f"command failed (exit {code}); log: {log}")

def board():
    """Status of every experiment in the shared Drive state."""
    rows = []
    for p in sorted(glob.glob(f"{STATE_DIR}/results/*/*/status.json")):
        s = json.load(open(p))
        age = (time.time() - s.get("updated_unix", 0)) / 60
        prog = f"{s.get('update', s.get('step', ''))}/{s.get('of', '')}" if s["state"] == "running" else ""
        rows.append(f"{s['state']:8s} {s['experiment_id']:48s} {s.get('worker',''):8s} {age:6.1f} min ago  {prog}")
    print("
".join(rows) if rows else "no experiments yet")

board()

## Task 1 — DPO

Order: smoke test → standard (train + eval) → β forks → length study → summary.
Approximate T4 times are estimates, not measurements: smoke ≈ 5 min, standard train ≈ 40–70 min,
each eval ≈ 10–15 min, each β fork ≈ 20 min + eval, length-balanced ≈ same as standard.
Different cells can run on different accounts at the same time.

In [ ]:
# Smoke test: 2 optimizer updates + tiny evaluation. Catches crashes before spending an hour.
run("python -m task1_dpo.train --run-name smoke --max-examples 32", "task1_smoke_train")
run("python -m task1_dpo.evaluate --adapter outputs/task1_dpo/smoke --name smoke --limit 4", "task1_smoke_eval")

In [ ]:
# Step 1: standard DPO, one epoch from the original initialization (its adapter is the Task 4 DPO policy)
run("python -m task1_dpo.train --config configs/dpo.yaml --run-name standard", "task1_train_standard")

In [ ]:
run("python -m task1_dpo.evaluate --config configs/dpo.yaml --adapter outputs/task1_dpo/standard --name standard", "task1_eval_standard")

In [ ]:
# Additional reference point (not a required condition): the untouched SFT policy under the same protocol
run("python -m task1_dpo.evaluate --config configs/dpo.yaml --name sft", "task1_eval_sft")

In [ ]:
# Step 2: beta forks — one per cell so they can be split across accounts
run("python -m task1_dpo.ablate_beta --config configs/dpo.yaml --only 0.03", "task1_beta003")

In [ ]:
run("python -m task1_dpo.ablate_beta --config configs/dpo.yaml --only 0.10", "task1_beta010")

In [ ]:
run("python -m task1_dpo.ablate_beta --config configs/dpo.yaml --only 0.30", "task1_beta030")

In [ ]:
# Step 3: length-balanced DPO + comparison (needs the standard eval to be done)
run("python -m task1_dpo.analyze_length --config configs/dpo.yaml", "task1_length")

In [ ]:
run("python -m task1_dpo.summarize --config configs/dpo.yaml", "task1_summary")

## Return results to GitHub
Zips `results/` (small JSON/CSV/log files only — adapters stay in `outputs/` on Drive) and downloads it.
Unzip it into the repo's `results/` folder on your PC and ask Claude to validate and commit.

In [ ]:
import shutil
from google.colab import files
stamp = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
archive = shutil.make_archive(f"/content/pa2_results_{stamp}", "zip", f"{STATE_DIR}/results")
print(archive, round(os.path.getsize(archive) / 2**20, 2), "MB")
files.download(archive)